In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os
from pathlib import Path
from datetime import datetime

project_root = Path.cwd().parent.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.features.features_v1 import *
from src.features.features_v2 import *
from src.pipeline.calculate_evs import *
from src.utils.helper_functions import *
from src.utils.team_info import teamStarPlayer, projectedStartingFive, mainStartingFive

### Update projected starting lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/utils/team_info.py
Updated 14 teams with confirmed lineups


### Load Model

### Load Player Data and Bookmaker Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

if us_file is None:
    raise FileNotFoundError(f"No NBA_US file found for {today}")
if dfs_file is None:
    raise FileNotFoundError(f"No NBA_DFS file found for {today}")

s26 = pd.read_csv('data/processed/training/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')
usData = pd.read_csv(us_file)
dfsData = pd.read_csv(dfs_file)

print(f"Loaded: {us_file.name}")
print(f"Loaded: {dfs_file.name}")
dfsData.head()

Loaded: NBA_US_20251207_124108.csv
Loaded: NBA_DFS_20251207_124008.csv


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,Underdog,player_points,Scottie Barnes,Over,19.5,-137,2025-12-07,2025-12-07T20:39:54Z,2025-12-07 12:40:08
1,Underdog,player_points,Scottie Barnes,Under,19.5,-137,2025-12-07,2025-12-07T20:39:54Z,2025-12-07 12:40:08
2,Underdog,player_points,Jaylen Brown,Over,29.5,-137,2025-12-07,2025-12-07T20:39:54Z,2025-12-07 12:40:08
3,Underdog,player_points,Jaylen Brown,Under,29.5,-137,2025-12-07,2025-12-07T20:39:54Z,2025-12-07 12:40:08
4,Underdog,player_points,Brandon Ingram,Over,22.5,-137,2025-12-07,2025-12-07T20:39:54Z,2025-12-07 12:40:08


In [4]:
from src.features.feature_engine import FeatureEngine

engine = FeatureEngine({
    "min_model": "src/models/saved/min_model.pkl",
    "usg_model": "src/models/saved/usg_model.pkl",
    "fga_model": "src/models/saved/fga_model.pkl",
    "ngboost_model_wrapper": "src/models/saved/pts_model_wrapper.pkl"
})

/Users/alexgonzalez/Documents/NBA-Prop-Predictor/nba_model/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Top EVs for 2 leg bets

### Underdog picks

In [5]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogPairs = calculate2LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2','ODDS 1', 'ODDS 2','PREDICTION 1', 'PREDICTION 2', 'MODEL_PROB 1', 'MODEL_PROB 2', 'SIDE 1', 'SIDE 2', 'PARLAY_PROB', 'PARLAY_ODDS', 'EV_PERCENT', 'KELLY_QUARTER']]
underdogPairs.to_csv('data/props/ev_analysis/underdogPairs.csv', index=False)
underdogPairs

Computing predictions for 59 players...
[MIN] No data found for Vincent Williams Jr.
Found 58 valid players
Generated 1383 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,MODEL_PROB 1,MODEL_PROB 2,SIDE 1,SIDE 2,PARLAY_PROB,PARLAY_ODDS,EV_PERCENT,KELLY_QUARTER
869,KJ Simpson,VJ Edgecombe,11.5,10.5,-120,110,3.09,14.51,0.912,0.806,under,over,0.735,285,183.16,0.1607
823,Spencer Jones,Buddy Hield,6.5,12.5,-137,-110,1.21,5.40,0.923,0.824,under,under,0.760,230,150.80,0.1639
385,Ja'Kobe Walter,Quinten Post,7.5,10.5,104,-113,2.91,3.86,0.765,0.816,under,under,0.624,285,140.30,0.1231
655,Brandon Miller,Jalen Williams,21.5,23.5,-106,-105,15.33,16.70,0.753,0.746,under,under,0.562,279,112.83,0.1011
730,Cameron Johnson,Ace Bailey,13.5,12.5,100,-115,8.83,6.91,0.719,0.772,under,under,0.555,274,107.51,0.0981
505,Jordan Walsh,Deni Avdija,7.5,25.5,-119,-115,3.88,20.30,0.717,0.711,under,under,0.510,244,75.35,0.0772
596,Nikola Jokić,Jimmy Butler III,29.5,21.5,-115,-113,23.79,16.52,0.693,0.693,under,under,0.480,252,69.00,0.0684
775,Peyton Watson,Santi Aldama,12.5,11.5,-112,-105,8.53,13.00,0.681,0.646,under,over,0.440,270,62.75,0.0581
1252,Brandin Podziemski,Keyonte George,9.5,21.5,-104,-107,10.57,17.69,0.618,0.647,over,under,0.400,279,51.59,0.0462
1113,Cam Spencer,Aaron Wiggins,11.5,15.5,100,-104,8.49,17.63,0.605,0.634,under,over,0.384,292,50.36,0.0431


### Prizepicks picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

prizepicksPairs = calculate2LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

prizepicksPairs = prizepicksPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2','ODDS 1', 'ODDS 2','PREDICTION 1', 'PREDICTION 2', 'MODEL_PROB 1', 'MODEL_PROB 2', 'SIDE 1', 'SIDE 2', 'PARLAY_PROB', 'PARLAY_ODDS', 'EV_PERCENT', 'KELLY_QUARTER']]
prizepicksPairs.to_csv('data/props/ev_analysis/prizepicksPairs.csv', index=False)
prizepicksPairs

Computing predictions for 81 players...
[MIN] No data found for Vincent Williams Jr.
Found 80 valid players
Generated 2649 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,MODEL_PROB 1,MODEL_PROB 2,SIDE 1,SIDE 2,PARLAY_PROB,PARLAY_ODDS,EV_PERCENT,KELLY_QUARTER
1488,KJ Simpson,Jake LaRavia,11.5,4.5,-120,-122,3.09,9.42,0.912,0.906,under,over,0.826,234,176.02,0.1881
726,Ja'Kobe Walter,Buddy Hield,7.5,12.5,104,-110,2.91,5.40,0.765,0.824,under,under,0.630,289,144.99,0.1254
1714,Tidjane Salaün,Quinten Post,6.5,10.5,-105,-113,2.17,3.86,0.789,0.816,under,under,0.644,268,137.12,0.1279
1214,Brandon Miller,Kenrich Williams,21.5,7.5,-106,-122,15.33,2.54,0.753,0.837,under,under,0.630,254,123.15,0.1212
2597,VJ Edgecombe,Jalen Williams,10.0,23.5,-137,-105,14.51,16.70,0.837,0.746,over,under,0.624,238,110.89,0.1165
1352,Cameron Johnson,Ace Bailey,13.5,12.5,100,-115,8.83,6.91,0.719,0.772,under,under,0.555,274,107.51,0.0981
436,Anfernee Simons,Kyle Filipowski,11.5,10.5,100,-107,14.38,6.19,0.717,0.700,over,under,0.502,287,94.29,0.0821
1805,Deni Avdija,Deandre Ayton,25.5,14.5,-115,104,20.30,16.37,0.711,0.651,under,over,0.463,281,76.51,0.0681
658,Jordan Walsh,Jimmy Butler III,7.5,21.5,-119,-113,3.88,16.52,0.717,0.693,under,under,0.497,247,72.38,0.0733
1116,Nikola Jokić,Jaylin Williams,29.5,8.0,-115,-137,23.79,3.86,0.693,0.739,under,under,0.512,223,65.23,0.0731


## 3 leg parlay

### Underdog picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points') ]

underdogTrios = calculate3LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'ODDS 1', 'ODDS 2', 'ODDS 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL_PROB 1', 'MODEL_PROB 2', 'MODEL_PROB 3', 'SIDE 1', 'SIDE 2', 'SIDE 3', 'PARLAY_PROB', 'PARLAY_ODDS', 'EV_PERCENT', 'KELLY_QUARTER']]
underdogTrios.to_csv('data/props/ev_analysis/underdogTrios.csv', index=False)
underdogTrios.head()

Computing predictions for 59 players...
[MIN] No data found for Vincent Williams Jr.
Found 58 valid players
Generated 17342 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,ODDS 1,ODDS 2,ODDS 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL_PROB 1,MODEL_PROB 2,MODEL_PROB 3,SIDE 1,SIDE 2,SIDE 3,PARLAY_PROB,PARLAY_ODDS,EV_PERCENT,KELLY_QUARTER
14270,KJ Simpson,Buddy Hield,VJ Edgecombe,11.5,12.5,10.5,-120,-110,110,3.09,5.40,14.51,0.912,0.824,0.806,under,under,over,0.606,635,345.32,0.1360
6623,Ja'Kobe Walter,Spencer Jones,Quinten Post,7.5,6.5,10.5,104,-137,-113,2.91,1.21,3.86,0.765,0.923,0.816,under,under,under,0.576,565,282.92,0.1252
10950,Brandon Miller,Deni Avdija,Jalen Williams,21.5,25.5,23.5,-106,-115,-105,15.33,20.30,16.70,0.753,0.711,0.746,under,under,under,0.399,609,183.22,0.0752
8923,Jordan Walsh,Cameron Johnson,Ace Bailey,7.5,13.5,12.5,-119,100,-115,3.88,8.83,6.91,0.717,0.719,0.772,under,under,under,0.398,588,173.54,0.0738
10504,Nikola Jokić,Santi Aldama,Jimmy Butler III,29.5,11.5,21.5,-115,-105,-113,23.79,13.00,16.52,0.693,0.646,0.693,under,over,under,0.310,588,113.45,0.0482


### Prizepicks picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

triosPrizepicks = calculate3LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'ODDS 1', 'ODDS 2', 'ODDS 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL_PROB 1', 'MODEL_PROB 2', 'MODEL_PROB 3', 'SIDE 1', 'SIDE 2', 'SIDE 3', 'PARLAY_PROB', 'PARLAY_ODDS', 'EV_PERCENT', 'KELLY_QUARTER']]
triosPrizepicks.to_csv('data/props/ev_analysis/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Computing predictions for 81 players...
[MIN] No data found for Vincent Williams Jr.
Found 80 valid players
Generated 46456 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,ODDS 1,ODDS 2,ODDS 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL_PROB 1,MODEL_PROB 2,MODEL_PROB 3,SIDE 1,SIDE 2,SIDE 3,PARLAY_PROB,PARLAY_ODDS,EV_PERCENT,KELLY_QUARTER
34286,KJ Simpson,Buddy Hield,Jake LaRavia,11.5,12.5,4.5,-120,-110,-122,3.09,5.40,9.42,0.912,0.824,0.906,under,under,over,0.681,537,333.65,0.1553
18001,Ja'Kobe Walter,Tidjane Salaün,Quinten Post,7.5,6.5,10.5,104,-105,-113,2.91,2.17,3.86,0.765,0.789,0.816,under,under,under,0.493,651,269.96,0.1037
29360,Brandon Miller,VJ Edgecombe,Kenrich Williams,21.5,10.0,7.5,-106,-137,-122,15.33,14.51,2.54,0.753,0.837,0.837,under,over,under,0.528,512,222.86,0.1088
9743,Anfernee Simons,Cameron Johnson,Jalen Williams,11.5,13.5,23.5,100,100,-105,14.38,8.83,16.70,0.717,0.719,0.746,over,under,under,0.385,681,200.39,0.0736
40001,Deni Avdija,Deandre Ayton,Ace Bailey,25.5,14.5,12.5,-115,104,-115,20.30,16.37,6.91,0.711,0.651,0.772,under,over,under,0.357,613,154.86,0.0632
